# S4.1 - Join types
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** The join type decides which rows survive when keys do not match

In [0]:
# ============================================================
# S4.1 - Cell 2
# Join Types: inner, left, right, full, cross,
#             left_semi, left_anti
#
# Goal:
# 1. Learn the behavior of each major join type.
# 2. Predict row counts on controlled data.
# 3. Compare predictions with actual Spark results.
# 4. See how unmatched rows behave.
# 5. Use left_anti to identify orphan records.
# 6. Repeat the exercise on samples.tpch.
#
# Scope:
# Join types only.
#
# Deferred:
# - multi-column / complex joins -> S4.2
# - broadcast joins            -> S4.3
# - physical join strategies   -> S4.4
# - duplicate/cardinality      -> S4.5
# ============================================================

print("=== S4.1: JOIN TYPES ===")
print()


# ------------------------------------------------------------
# Step 1 - Controlled Source Data
#
# Orders:
#
# 101 -> customer 1
# 102 -> customer 2
# 103 -> customer 9 (missing from customers)
# 104 -> customer 1
#
# Customers:
#
# 1 -> Asha
# 2 -> Ben
# 3 -> Chen (has no orders)
# ------------------------------------------------------------

orders = spark.createDataFrame(
    [
        (101, 1),
        (102, 2),
        (103, 9),
        (104, 1)
    ],
    "order_id INT, cust_id INT"
)

customers = spark.createDataFrame(
    [
        (1, "Asha"),
        (2, "Ben"),
        (3, "Chen")
    ], "cust_id INT, name STRING"
)

print("=== STEP 1: ORDERS ===")
orders.orderBy("order_id").show()

print("=== STEP 1: CUSTOMERS ===")
customers.orderBy("cust_id").show()

In [0]:
# ------------------------------------------------------------
# Step 2 - Predict and Validate Every Join Type
# ------------------------------------------------------------

predicted = {

    # Matching order/customer pairs only:
    # orders 101, 102, 104
    "inner": 3,

    # Preserve all four orders.
    # Order 103 gets NULL for customer columns.
    "left": 4,

    # Preserve every customer.
    # Asha matches two orders, Ben one, Chen none.
    "right": 4,

    # All matched and unmatched records from both sides.
    "full": 5,

    # Left rows having a customer match.
    # Returns orders columns only.
    "left_semi": 3,

    # Left rows having no customer match.
    # Order 103 only.
    "left_anti": 1
}

print("=== STEP 2: JOIN TYPES ===")

actual = {}

for how, expected in predicted.items():
    result = (orders.join(customers, on = "cust_id", how = how))

    actual_count = (result.count())

    actual[how] = (actual_count)

    print(
        f"{how : <10} "
        f"predicted = {expected : <2} "
        f"actual = {actual_count : <2} "
        f"match = {expected == actual_count} "
        f"columns = {result.columns}")
    
# ------------------------------------------------------------
# CROSS JOIN
#
# Every order pairs with every customer:
#
# 4 orders x 3 customers = 12 rows
# ------------------------------------------------------------

cross_count = (
    orders
    .crossJoin(customers).count())
    
print(
    f"{'cross' : <10} "
    f"predicted = 12 "
    f"actual={cross_count : <2} "
    f"match={cross_count == 12} "
    f"(4 orders x 3 customers)"
)
print()

In [0]:
# ------------------------------------------------------------
# Step 3 - Inspect LEFT JOIN
#
# A LEFT JOIN preserves all rows from orders.
#
# Order 103 has customer 9.
# Customer 9 does not exist.
#
# Therefore:
#
# order_id = 103
# name     = NULL
# ------------------------------------------------------------

print("=== STEP 3: LEFT JOIN ROWS ===")

(orders.join(customers,"cust_id", "left")
 .orderBy("order_id").show())

In [0]:
# ------------------------------------------------------------
# Step 3b - LEFT ANTI
#
# LEFT ANTI answers:
#
# Which orders have NO matching customer?
#
# Expected:
#
# order 103
# ------------------------------------------------------------

print("=== STEP 3b: LEFT ANTI ===")


(orders.join(customers, "cust_id","left_anti").show())


In [0]:
# ------------------------------------------------------------
# Step 4 - Real Data: samples.tpch
#
# Orders:
# o_orderkey
# o_custkey
#
# Customer:
# c_custkey
# c_name
#
# Key names differ, so an equality condition is used.
#
# Detailed complex-condition joins are covered in S4.2.
# ------------------------------------------------------------

o = (
    spark.table(
        "samples.tpch.orders"
    )
    .select(
        "o_orderkey",
        "o_custkey"
    )
)


c = (spark.table("samples.tpch.customer")
    .select("c_custkey", "c_name"))

condition = (o["o_custkey"] == c["c_custkey"])

n_orders = (o.count())

n_inner = (
    o
    .join(
        c,
        condition,
        "inner"
    )
    .count()
)


n_left = (o.join(c, condition,"left").count())

n_anti = (o.join(c, condition, "left_anti").count())

print()
print("=== STEP 4: REAL DATA ===")

print(f"orders rows             : " f"{n_orders:,}")

print(f"inner join rows         : " f"{n_inner:,}")

print(f"left join rows          : " f"{n_left:,}")

print(f"left_anti (no customer) : " f"{n_anti:,}")


print(f"left join == orders rows: " f"{n_left == n_orders}")


In [0]:
# This is an observed reconciliation for this dataset.
# S4.5 will explain how duplicate join keys can change
# inner/outer join row counts.

print(
    f"inner + anti == orders "
    f"(this dataset): "
    f"{n_inner + n_anti == n_orders}"
)

# S4.1 — Key Takeaways: Join Types

## 1. What the Join Type Decides

The join type decides which rows survive when join keys do not match.

| Join | Keeps | Columns returned |
|---|---|---|
| `inner` | rows matching on both sides | both tables |
| `left` | all left rows; unmatched right columns become NULL | both tables |
| `right` | all right rows; unmatched left columns become NULL | both tables |
| `full` | matched and unmatched rows from both sides | both tables |
| `cross` | every left row paired with every right row | both tables |
| `left_semi` | left rows that have at least one match | left table only |
| `left_anti` | left rows that have no match | left table only |

---

## 2. Orders With a Missing Customer

Use a `left` join when every order must remain even if the customer is missing.

Example:

```text
order 103
cust_id = 9
customer does not exist
```

The left join returns:

```text
order_id = 103
cust_id  = 9
name     = NULL
```

An `inner` join would remove order 103 because no matching customer exists.

This creates a similar silent row-loss risk to other operations that remove records without raising an error, although the mechanism is different.

`left_anti` is useful when we specifically want:

```text
orders with no matching customer
```

---

## 3. Cross Join

A cross join creates every possible left/right combination.

For our controlled data:

```text
4 orders
×
3 customers
=
12 rows
```

Therefore:

```text
cross join rows
=
left rows × right rows
```

Use cross joins carefully because row counts can grow very quickly.

---

## 4. Reconciliation

Always understand the expected row-count behaviour after a join.

For a left join:

```text
If every left row matches at most one right row
→ left join row count = left source row count
```

If:

```text
left join rows > left source rows
```

then at least one left row matched multiple right-side rows and fan-out occurred.

Detailed duplicate-key and join-cardinality behaviour is covered in **S4.5**.

For our current TPC-H dataset we also checked:

```text
inner rows + left_anti rows
=
orders rows
```

This is an observed reconciliation for this dataset, not a universal join rule.

---

## 5. Controlled Dataset Checkpoint

Source:

```text
orders    = 4 rows
customers = 3 rows
```

Observed join counts:

```text
inner      predicted 3   actual 3
left       predicted 4   actual 4
right      predicted 4   actual 4
full       predicted 5   actual 5
left_semi  predicted 3   actual 3
left_anti  predicted 1   actual 1
cross      predicted 12  actual 12
```

All predictions matched the actual results.

Therefore:

```text
match = False
→ none
```

---

## 6. Controlled LEFT JOIN Checkpoint

For order:

```text
order_id = 103
cust_id  = 9
```

there was no matching customer.

Observed:

```text
name = NULL
```

This confirms:

> A left join preserves the left-side row even when the right-side match is missing.

---

## 7. LEFT SEMI and LEFT ANTI

### LEFT SEMI

Returns left-side rows that have at least one match.

In our controlled example:

```text
101
102
104
```

It returns only columns from `orders`.

### LEFT ANTI

Returns left-side rows that have no match.

Observed:

```text
order_id = 103
cust_id  = 9
```

This makes `left_anti` useful for:

```text
orphan detection
missing reference records
data-quality checks
```

---

## 8. Observed on samples.tpch

For:

```text
samples.tpch.orders
vs
samples.tpch.customer
```

Observed:

```text
orders rows              = 7,500,000
inner join rows          = 7,500,000
left join rows           = 7,500,000
left_anti (no customer)  = 0

left join == orders rows = True
```

Therefore, for this join:

```text
Every order found a matching customer.
```

Also:

```text
left_anti = 0
```

means no orphan orders were found.

The left join did not increase the order row count, so no customer-side fan-out occurred for these order/customer matches.

---

## 9. Checkpoint Questions

### Q1. Did any controlled join show `match=False`?

No.

All predicted and actual row counts matched.

---

### Q2. What was `name` for order 103 after the left join?

```text
NULL
```

Because customer `9` did not exist.

---

### Q3. Which join types return only left-side columns?

```text
left_semi
left_anti
```

---

### Q4. What did the TPC-H `left_anti` result tell us?

```text
left_anti = 0
```

Therefore no order in this dataset was missing its customer.

---

### Q5. If we must preserve orders whose customer is missing, should we use inner or left join?

Use a **left join**, because it preserves every order and represents a missing customer with NULL right-side columns.

---

## 10. Core Understanding

```text
Need only matching rows?
→ INNER

Need every left row?
→ LEFT

Need every right row?
→ RIGHT

Need everything from both sides?
→ FULL

Need every possible combination?
→ CROSS

Need left rows that have a match?
→ LEFT SEMI

Need left rows that do not have a match?
→ LEFT ANTI
```

**S4.1 — Join Types COMPLETED.**